In [ ]:
# Install dependency yang dibutuhkan Playwright/Chromium
!apt-get update -qq

!apt-get install -y \
    libatk1.0-0 \
    libatk-bridge2.0-0 \
    libcups2 \
    libxkbcommon0 \
    libxcomposite1 \
    libxdamage1 \
    libxfixes3 \
    libxrandr2 \
    libgbm1 \
    libasound2 \
    libpangocairo-1.0-0 \
    libpango-1.0-0 \
    libcairo2 \
    libatspi2.0-0 \
    libgtk-3-0

!node -v
!npm -v

In [ ]:
# Cell 1 — Imports
import pandas as pd
import os
import time
from datetime import datetime, timedelta

In [ ]:
# ==========================================
# 2. AMBIL TOKEN DARI COLAB SECRETS
# ==========================================
from google.colab import userdata

twitter_auth_token = userdata.get("twitter_auth_token")

if twitter_auth_token is None:
    raise ValueError(
        "Secret 'twitter_auth_token' tidak ditemukan."
    )

print("Token berhasil dimuat.")

In [ ]:
# Cell 2 — Keywords (diperluas)
negative_keywords = [
    # Kata kasar
    "goblok", "tolol", "idiot", "bangsat", "kampret",
    "anjing", "bajingan", "brengsek", "bodoh", "dungu",
    # Ekspresi negatif umum
    "jelek", "buruk", "gagal", "payah", "sampah",
    "menyebalkan", "menjengkelkan", "mengecewakan",
    "benci", "marah", "kesal", "frustrasi", "stress",
    # Cyberbullying / toxic
    "bullying", "bully", "hina", "rendah", "memalukan",
]

neutral_keywords = [
    # Aktivitas harian
    "cuaca", "sekolah", "kampus", "belajar", "jalan",
    "kereta", "makan", "olahraga", "tidur", "bangun",
    # Topik umum
    "berita", "informasi", "update", "kabar", "info",
    "hari ini", "kemarin", "besok", "minggu ini",
    "kerja", "kantor", "meeting", "tugas", "kuliah",
    # Tempat / aktivitas netral
    "belanja", "pasar", "mall", "rumah", "kota",
]

positive_keywords = [
    # Pujian
    "bagus", "hebat", "keren", "mantap", "luar biasa",
    "terbaik", "membanggakan", "sukses", "berhasil",
    # Emosi positif
    "senang", "bahagia", "gembira", "bersyukur", "bangga",
    "semangat", "termotivasi", "terinspirasi",
    # Apresiasi
    "terimakasih", "terima kasih", "salut", "kagum",
    "recommended", "rekomen", "worth it", "memuaskan",
]

keyword_groups = {
    "negatif": negative_keywords,
    "netral":  neutral_keywords,
    "positif": positive_keywords
}

# Info ringkas
print("Keyword groups siap:")
for label, kws in keyword_groups.items():
    print(f"  [{label}] {len(kws)} keyword")
print(f"  Total: {sum(len(v) for v in keyword_groups.values())} keyword")

In [ ]:
# Cell — Konfigurasi (jalankan sebelum cell scraping)
STOP_PER_RUNTIME = 1000     # batas data yang diambil per sesi runtime, sesuaikan
LIMIT_PER_QUERY  = 200      # batas tweet per keyword per request, sesuaikan
TARGET_DATA      = 10000




In [ ]:
# Cell — Setup checkpoint dari dataset gabungan
import os
import pandas as pd

CHECKPOINT_FILE = "/content/dataset_final.csv"  # pakai hasil gabungan sebagai checkpoint
TARGET_DATA = 10000

def load_checkpoint():
    if os.path.exists(CHECKPOINT_FILE):
        df = pd.read_csv(CHECKPOINT_FILE)
        print(f"[CHECKPOINT] Memuat {len(df)} data dari {CHECKPOINT_FILE}")
        return df
    else:
        print("[CHECKPOINT] Belum ada file, mulai dari kosong.")
        return pd.DataFrame()

dataset = load_checkpoint()
print(f"Sisa target: {max(0, TARGET_DATA - len(dataset))}")

In [ ]:
# Cell 6 (modifikasi) — Main Scraping, lanjutkan dari dataset_final.csv
import random
import time

dataset      = load_checkpoint()
data_run_ini = 0

# Tentukan kolom teks untuk cek duplikat
text_col = "full_text" if "full_text" in dataset.columns else "tweet"

keyword_queues = {label: kws[:] for label, kws in keyword_groups.items()}
for label in keyword_queues:
    random.shuffle(keyword_queues[label])

print(f"[START] Data existing: {len(dataset)} | Target: {TARGET_DATA}")

max_kw = max(len(v) for v in keyword_queues.values())

for i in range(max_kw):

    if data_run_ini >= STOP_PER_RUNTIME:
        print(f"\nBatas runtime ({STOP_PER_RUNTIME}) tercapai.")
        break

    if len(dataset) >= TARGET_DATA:
        print(f"\nTarget {TARGET_DATA} tercapai!")
        break

    for label, kws in keyword_queues.items():

        if i >= len(kws):
            continue
        if data_run_ini >= STOP_PER_RUNTIME:
            break
        if len(dataset) >= TARGET_DATA:
            break

        keyword     = kws[i]
        output_file = f"keyword_{keyword}.csv"

        query = (
            f'"{keyword}" '
            f'lang:id '
            f'-is:retweet'
        )

        print(f"\n  [{label}] Keyword : {keyword}")

        !npx --yes tweet-harvest@2.6.1 \
            -o "{output_file}" \
            -s "{query}" \
            -l {LIMIT_PER_QUERY} \
            --token {twitter_auth_token}

        time.sleep(5)

        if os.path.exists(output_file):
            try:
                temp_df = pd.read_csv(output_file)

                if len(temp_df) > 0:
                    temp_df["label"]   = label
                    temp_df["keyword"] = keyword

                    cur_text_col = "full_text" if "full_text" in temp_df.columns else "tweet"

                    # Anti-duplikat langsung terhadap dataset gabungan
                    if len(dataset) > 0 and text_col in dataset.columns and cur_text_col in temp_df.columns:
                        existing_texts = set(dataset[text_col].dropna())
                        before = len(temp_df)
                        temp_df = temp_df[~temp_df[cur_text_col].isin(existing_texts)]
                        dup_removed = before - len(temp_df)
                        if dup_removed > 0:
                            print(f"  ⚠ {dup_removed} duplikat (vs dataset_final) dihapus")

                    # Hapus duplikat internal dalam batch baru ini juga
                    if cur_text_col in temp_df.columns:
                        temp_df.drop_duplicates(subset=[cur_text_col], inplace=True)

                    if len(temp_df) > 0:
                        dataset = pd.concat([dataset, temp_df], ignore_index=True, sort=False)
                        dataset.reset_index(drop=True, inplace=True)

                        data_baru     = len(temp_df)
                        data_run_ini += data_baru

                        # Simpan progres langsung tiap keyword selesai (biar aman kalau runtime putus)
                        dataset.to_csv(CHECKPOINT_FILE, index=False)

                        print(f"  → {data_baru} tweet baru | run ini: {data_run_ini}/{STOP_PER_RUNTIME} | total: {len(dataset)}")
                    else:
                        print(f"  → Semua data sudah ada (duplikat), skip.")
                else:
                    print(f"  → 0 tweet didapat")

                os.remove(output_file)

            except Exception as e:
                print(f"  Error pada {keyword}: {e}")

# === Simpan checkpoint akhir ===
dataset.to_csv(CHECKPOINT_FILE, index=False)

print("\n" + "=" * 60)
print(f"Total data keseluruhan : {len(dataset)} / {TARGET_DATA}")
print(f"Data didapat run ini   : {data_run_ini}")
print(f"Sisa target            : {max(0, TARGET_DATA - len(dataset))}")
print("=" * 60)
print(f"Disimpan ke {CHECKPOINT_FILE}")

In [ ]:
# Cell — Gabungkan semua CSV hasil scraping menjadi satu dataset_final.csv
import pandas as pd
import glob
import os

DATA_DIR = "/content/tweets-data"
OUTPUT_FILE = "dataset_final.csv"

# Ambil semua file csv di folder (termasuk subfolder kalau ada)
csv_files = glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True)

print(f"Ditemukan {len(csv_files)} file CSV:")
for f in csv_files:
    print(" -", f)

all_dfs = []

for file in csv_files:
    try:
        df = pd.read_csv(file)
        if len(df) == 0:
            print(f"  [skip] {file} kosong")
            continue

        # Tambahkan info keyword dari nama file jika kolom belum ada
        if "keyword" not in df.columns:
            base = os.path.basename(file)
            kw = base.replace("keyword_", "").replace("kw_", "").replace(".csv", "")
            df["keyword"] = kw

        df["source_file"] = os.path.basename(file)
        all_dfs.append(df)
        print(f"  [ok] {file} -> {len(df)} baris")

    except Exception as e:
        print(f"  [error] Gagal baca {file}: {e}")

if not all_dfs:
    raise SystemExit("Tidak ada data yang berhasil dibaca.")

# Gabungkan semua dataframe (tanpa menghapus duplikat, biarkan apa adanya)
merged = pd.concat(all_dfs, ignore_index=True, sort=False)
print(f"\nTotal baris gabungan: {len(merged)} baris (duplikat tidak dihapus)")

merged.to_csv(OUTPUT_FILE, index=False)
print(f"\n✅ Disimpan ke {OUTPUT_FILE} | Total akhir: {len(merged)} baris")